# Cobertura e qualidade das DFP de 2024

ImpactUFSCar — Área de Pesquisa & Mercado

Qual é a cobertura contábil disponível para estudar qualidade operacional e estresse financeiro? O notebook lê um snapshot gerado pelo pipeline em `data/interim`, mantém as versões de cada entrega e analisa os escopos consolidado (`con`) e individual (`ind`) separadamente.

Na raiz do repositório, execute `uv run --locked quantdistress-dfp --ano 2024` para gerar o snapshot. Se houver múltiplos snapshots para o ano, defina `QUANTDISTRESS_DFP_SNAPSHOT` com o diretório do snapshot escolhido. O caminho pode ser absoluto ou relativo à raiz do repositório.

In [ ]:
import json
import os
import re
from hashlib import file_digest
from pathlib import Path

import pandas as pd
from IPython.display import display

from quantdistress.evaluation.quality import (
    CHAVE_ENTREGA,
    cobertura_por_demonstracao,
    cobertura_por_escopo,
    conciliacao_patrimonial,
    integridade_por_demonstracao,
    rotulos_contas_candidatas,
)

ano = 2024
pasta_atual = Path.cwd()
raiz = next(
    (pasta for pasta in (pasta_atual, *pasta_atual.parents) if (pasta / "pyproject.toml").is_file()),
    None,
)
if raiz is None:
    raise FileNotFoundError("Abra o notebook pelo repositório do QuantDistress Brasil.")

caminho_configurado = os.environ.get("QUANTDISTRESS_DFP_SNAPSHOT")
if caminho_configurado:
    snapshot = (raiz / caminho_configurado).resolve()
else:
    pasta_ano = raiz / "data" / "interim" / "dfp" / str(ano)
    manifestos = [
        arquivo for arquivo in pasta_ano.glob("*/manifesto.json")
        if re.fullmatch(r"[0-9a-f]{64}", arquivo.parent.name)
    ]
    if len(manifestos) != 1:
        raise RuntimeError(
            f"Esperado um snapshot local para {ano}, encontrados {len(manifestos)}. "
            "Gere o snapshot ou defina QUANTDISTRESS_DFP_SNAPSHOT."
        )
    snapshot = manifestos[0].parent

manifesto = json.loads((snapshot / "manifesto.json").read_text(encoding="utf-8"))
if manifesto["ano"] != ano:
    raise ValueError(f"Selecione um snapshot para {ano}.")


def ler_tabela(nome):
    especificacao = manifesto["tabelas"][nome]
    arquivo = snapshot / especificacao["arquivo"]
    with arquivo.open("rb") as entrada:
        if file_digest(entrada, "sha256").hexdigest() != especificacao["sha256"]:
            raise ValueError(f"Hash divergente para {arquivo.name}.")
    dados = pd.read_parquet(arquivo)
    if len(dados) != especificacao["linhas"]:
        raise ValueError(f"Contagem de linhas divergente para {arquivo.name}.")
    return dados


fatos = ler_tabela("fatos")
entregas = ler_tabela("entregas")
with pd.option_context("display.max_colwidth", None):
    display(pd.Series(manifesto["fonte"], name="SNAPSHOT CVM"))
    display(pd.DataFrame(manifesto["tabelas"]).T)

## Entregas e datas

O identificador de entrega reúne CNPJ, código CVM, data de referência e versão. A contagem por versão usa as linhas do índice. A data de recebimento permite estudar as defasagens do índice. Uma análise histórica precisa confirmar quando a informação se tornou pública.

In [ ]:
chaves = entregas[CHAVE_ENTREGA].drop_duplicates()
fatos_por_chave = fatos[["ESCOPO", *CHAVE_ENTREGA]].drop_duplicates()
chaves_com_fatos = fatos_por_chave[CHAVE_ENTREGA].drop_duplicates()
display(
    pd.Series(
        {
            "LINHAS_INDICE": len(entregas),
            "EMISSORES_CD_CVM": entregas["CD_CVM"].nunique(),
            "CHAVES_ENTREGA": len(chaves),
            "CHAVES_COM_MULTIPLAS_VERSOES": chaves.groupby(CHAVE_ENTREGA[:-1]).size().gt(1).sum(),
            "CHAVES_COM_MULTIPLOS_REGISTROS": len(
                entregas.loc[entregas["CHAVE_ENTREGA_REPETIDA"], CHAVE_ENTREGA].drop_duplicates()
            ),
            "CHAVES_COM_FATOS": len(chaves_com_fatos),
            "CHAVES_DO_INDICE_SEM_FATOS": len(
                chaves.merge(chaves_com_fatos, on=CHAVE_ENTREGA, how="left", indicator=True)
                .loc[lambda dados: dados["_merge"].eq("left_only")]
            ),
            "RECEBIMENTO_MIN": entregas["DT_RECEB"].min(),
            "RECEBIMENTO_MAX": entregas["DT_RECEB"].max(),
        },
        name="VALOR",
    )
)
display(pd.crosstab(entregas["DT_REFER"].dt.date, entregas["VERSAO"], margins=True))
display(entregas["DT_RECEB"].dt.year.value_counts().sort_index().rename("LINHAS_INDICE"))

chave_sem_versao = CHAVE_ENTREGA[:-1]
versoes_maximas = chaves.groupby(chave_sem_versao)["VERSAO"].max().rename("VERSAO_MAXIMA")
versoes_observadas = fatos_por_chave.join(versoes_maximas, on=chave_sem_versao)
situacao_versao = pd.Series("fora do índice", index=versoes_observadas.index)
situacao_versao.loc[versoes_observadas["VERSAO"].lt(versoes_observadas["VERSAO_MAXIMA"])] = (
    "versão inferior à máxima no índice"
)
situacao_versao.loc[versoes_observadas["VERSAO"].eq(versoes_observadas["VERSAO_MAXIMA"])] = (
    "versão máxima no índice"
)
display(pd.crosstab(versoes_observadas["ESCOPO"], situacao_versao.rename("VERSAO_OBSERVADA")))

## Cobertura por demonstração

A primeira tabela conta linhas e chaves em cada arquivo, inclusive exercícios comparativos. A segunda mede a presença de demonstrações do último exercício para todas as chaves do índice, em cada escopo. `BPA_BPP_DRE_DFC` exige os dois balanços, DRE e pelo menos um método de DFC. São medidas de presença de blocos e não verificam se os insumos de um indicador são válidos.

No snapshot avaliado, as demonstrações estão disponíveis apenas na versão máxima de cada chave de emissor e data. O índice guarda as versões anteriores, mas a recuperação dos valores históricos depende de outras capturas ou dos documentos originais.

In [ ]:
display(cobertura_por_demonstracao(fatos))
display(cobertura_por_escopo(fatos, entregas).T)
display(fatos.groupby(["MOEDA", "ESCALA_MOEDA"], dropna=False).size().rename("LINHAS"))

## Exercícios e comparativos

A data de referência identifica a entrega. Início e fim do exercício identificam o período atribuído ao valor. A duração abaixo conta dias de calendário, incluindo as duas datas, e usa períodos distintos por classe de demonstração e chave de entrega.

In [ ]:
colunas_periodo = [
    *CHAVE_ENTREGA, "ESCOPO", "FAMILIA", "ORDEM_EXERC", "DT_INI_EXERC", "DT_FIM_EXERC",
]
fluxos = fatos.loc[fatos["FAMILIA"].isin(["DRE", "DFC_MD", "DFC_MI"]), colunas_periodo]
periodos = fluxos.drop_duplicates().assign(
    DIAS=lambda dados: (dados["DT_FIM_EXERC"] - dados["DT_INI_EXERC"]).dt.days + 1
)
periodos["DURACAO"] = periodos["DIAS"].map({365: "365 dias", 366: "366 dias"}).fillna(
    "Outra duração ou data ausente"
)
display(
    periodos.groupby(["ESCOPO", "FAMILIA", "ORDEM_EXERC", "DURACAO"], dropna=False)
    .size()
    .unstack(fill_value=0)
)
especiais = periodos.loc[~periodos["DIAS"].isin([365, 366])]
display(especiais["DIAS"].value_counts(dropna=False).sort_index().rename("PERIODOS"))

## Integridade e conciliação patrimonial

As contagens de repetição são de linhas afetadas, incluindo a primeira ocorrência. Um vínculo ambíguo impede atribuir a data de recebimento de uma entrega específica a esses fatos. As linhas permanecem disponíveis para auditoria.

A conciliação compara BPA `1` com BPP `2` dentro da mesma chave, escopo e período. Se as linhas repetidas têm os mesmos valores monetários e escala, o total participa uma vez. Totais concorrentes permanecem indeterminados. Em cada par comparável, convertemos ambos os valores para unidades da moeda informada e admitimos uma unidade da maior escala de publicação. Este é um teste de totais e tem alcance limitado a essa identidade.

In [ ]:
display(integridade_por_demonstracao(fatos))
pares = conciliacao_patrimonial(fatos)
display(pd.crosstab([pares["ESCOPO"], pares["ORDEM_EXERC"]], pares["SITUACAO"]))
display(
    pares.loc[pares["DIFERENCA_EM_UNIDADES"].notna()]
    .groupby("ESCOPO")["DIFERENCA_EM_UNIDADES"]
    .agg(lambda coluna: max(coluna.map(lambda diferenca: diferenca.copy_abs())))
    .rename("MAIOR_DIFERENCA_ABSOLUTA_EM_UNIDADES")
)

## Contas candidatas para a Sprint 02

A mesma codificação pode corresponder a conceitos diferentes entre layouts. A tabela registra a ocorrência de cada rótulo no último exercício. Valide as definições por setor, método e escopo antes de calcular indicadores. O valor de uma conta por ação, por exemplo, requer uma regra diferente das contas monetárias da mesma DRE.

In [ ]:
rotulos = rotulos_contas_candidatas(fatos).sort_values(
    ["ESCOPO", "FAMILIA", "CD_CONTA", "CHAVES_ENTREGA"],
    ascending=[True, True, True, False],
)
with pd.option_context("display.max_colwidth", None):
    display(rotulos.reset_index(drop=True))

Para o Quality Score, o mapeamento deve validar contas de receita, resultado operacional, caixa operacional e capital investido. Para o Financial Distress Score, as prioridades incluem ativo e passivo circulantes, empréstimos, despesa financeira e patrimônio líquido. São dois eixos de pesquisa. Uma auditoria de cobertura de um ano orienta o mapeamento, e a avaliação empírica requer séries históricas e eventos datados.